---
title: "Insights from a visual cortical column connectome"
subtitle: "Society for Neuroscience 2026 &middot; PSTR230.05"
author:
  - Bethanny Danskin
  - Chi Zhang
  - Erika Neace
  - Rachael Swanstrom
  - Forrest Collman
  - Nuno da Costa
  - Casey Schneider-Mizell
date: 2026-11-17
date-format: "D MMMM YYYY"
---

::: {.callout-note appearance="simple"}
This is an interactive page covering the material from the  SfN 2026 poster. Three of the figures are **interactive** — turn a cell type off in
the legend and the rest redraw — and a few supplemental panels that did not fit on the
board are included at the end of their section.

[Download the original poster as printed (PDF, 5.6 MB)](sfn_2026_poster.pdf)
:::

## About the data in this poster

**Versions.** The connectome is MICrONS `minnie65_public`. Every number on
this page is at materialization **v1926** unless it names another; **v1300** is
the state at the *Nature* 2025 release and is the baseline every "since
publication" comparison uses.

In [1]:
#| echo: False
#| label: setup
#| code-summary: "Setup: load the exported data"
import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import HTML, display

DATA = Path("data")
FIGURES = Path("figures")

# `plotly_mimetype` is what Quarto's jupyter engine picks up; the
# `notebook_connected` half writes a CDN script tag so the figures also work
# if a single .ipynb is opened on its own, away from the rendered site.
pio.renderers.default = "plotly_mimetype+notebook_connected"

PALETTE = json.loads((DATA / "palette.json").read_text(encoding="utf-8"))
COLOUR = PALETTE["label_colour"]
ORDER = PALETTE["label_order"]
SHOWN = PALETTE["display_name"]          # "Basket family" -> "Basket"
EXCITATORY = PALETTE["excitatory_order"]

somata = pd.read_csv(DATA / "somata.csv")
types = pd.read_csv(DATA / "type_matrix.csv")
totals = pd.read_csv(DATA / "type_totals.csv").set_index("group")
density = pd.read_csv(DATA / "reach_density.csv")
reach_meta = json.loads((DATA / "reach_meta.json").read_text(encoding="utf-8"))
graph = json.loads((DATA / "graph_stats.json").read_text(encoding="utf-8"))

# Groups in the poster's own order, restricted to those actually present.
GROUPS = [g for g in ORDER if g in set(somata.group)]

#: One Plotly layout for every interactive figure on the page, so they read as
#: a set rather than as three different plots that happen to share a palette.
LAYOUT = dict(
    template="simple_white",
    font=dict(family="Arial, Helvetica, sans-serif", size=14, color="#111111"),
    margin=dict(l=70, r=30, t=70, b=60),
    hoverlabel=dict(font_size=13),
)
LEGEND = dict(bgcolor="rgba(255,255,255,0.75)", bordercolor="#D4D9DB",
              borderwidth=1, itemsizing="constant")
CONFIG = {"displaylogo": False, "responsive": True,
          "modeBarButtonsToRemove": ["select2d", "lasso2d", "autoScale2d"]}


def layout(**overrides):
    """`LAYOUT` with per-figure overrides, merging the legend rather than
    replacing it.

    `update_layout(**LAYOUT, legend=...)` raises — LAYOUT would already carry
    a `legend` and Python sees two values for one keyword — and the obvious
    repair, dropping the shared legend styling, silently gives each figure a
    different one.
    """
    merged = dict(LAYOUT)
    merged.update(overrides)
    merged["legend"] = {**LEGEND, **overrides.get("legend", {})}
    return merged

# print(f"{len(somata):,} somata  |  {len(types)} type pairs  |  "
#       f"{int(density.synapses.sum()):,} output synapses")

In [2]:
#| label: headline
#| echo: false
#| code-summary: "The headline numbers, as on the board"
STATS = [
    ("1,355", "neurons in a 100 µm column"),
    ("153,178", "synapses between those neurons"),
    ("643,464", "connections from the column to any neuron"),
    ("1,384,465", "total unique connections in the 1 mm³ volume"),
]
display(HTML('<div class="statrow">' + "".join(
    f'<div class="stat"><span class="value">{v}</span>'
    f'<span class="label">{t}</span></div>' for v, t in STATS) + "</div>"))

## 01 — The MICrONS dataset

![](figures/panel01_volume.png){fig-alt="Rendering of every reconstructed
process in a cubic millimetre of mouse visual cortex, each cell a different
colour."}

A volume electron-microscopy reconstruction of roughly a cubic millimetre of
mouse visual cortex, spanning V1 and the higher visual areas lateral to it.
Every object in the render is a segmented process from the automated
segmentation; the colours are arbitrary and are there to separate neighbours,
not to encode type.

This reconstruction is the substrate for everything below. The automated segmentation gets
dendrites largely right but axons are often falsely-truncated or falsely-merged to other neurits. 
Section 02 describes the ongoing manual proofreading efforts to extend the axons.

*The MICrONS Consortium et al., Nature 2025.*
[doi:10.1038/s41586-025-08790-w](https://doi.org/10.1038/s41586-025-08790-w)

## 02 — Ongoing refinement

Proofreading is not a binary. A cell moves along a ladder — untouched, clean,
partially extended, fully extended — and where it sits determines which
questions it can answer. An unproofread axon is not a short axon; it is an
axon whose reconstruction truncated early and often includes falsely-merged axon from another source. 
Counting its synapses gives an answer that is largely unpredictive of its corrected reconstruction.

### Where the tracked population sits, and where it moved

![](figures/block02a_proofreading_flow.png){fig-alt="Sankey diagram of
proofreading status at v1300 flowing to status at v1926, for column cells and
for cells outside the column."}

Status at the *Nature* release (v1300) on the left, status now (v1926) on the
right, for the V1 column and for the tracked cells outside it. The column is
the campaign this poster is about: 1,355 cells, of which 541 are now fully
extended (prioritized by the least proofread to extend first) and 813 partially. 
Outside the column the work is broader and
shallower, and the interareal category — axons followed specifically across an
area border — exists only there.

Read the thickness, not the endpoints: most of the movement is
*partially → fully extended*, which is the expensive half of the ladder.

### What fully extending one neuron takes

::: {layout-ncol=2}
![](figures/block02b_neuron_296503.png){fig-alt="One Layer 4 intratelencephalic pyramidal cell with
dendrite in dark grey, the axon as reconstructed at publication in light grey,
and the axon added since in purple."}

![](figures/block02c_hours_296503.png){fig-alt="Cumulative output synapses
against cumulative hours worked for the same cell, rising from about 660 to
2,484 over 69 hours."}
:::

One cell, nucleus 296503, before and after. The dendrite (dark) was already
correct; the light grey axon is what the segmentation and earlier cleanup provided,
and the purple is what full extension added. Axon length goes 6.2 → 20.0 mm.

The right panel is the same cell against the clock. Output synapses rise from
659 to 2,484 over **69 hours** of human work — and the curve is close to
linear, meaning there is no easy plataeu at which we consider a cell representaitonally 
complete until it is complete within the volume.

### Output synapses added, across the whole column

![](figures/block02d_output_gains_column.png){fig-alt="Horizontal bars, one per
column neuron grouped by cell type, showing output synapses at publication in
grey and the synapses added since in the type's colour."}

Every column neuron, grouped by type, sorted within type. Grey is what was
there at the *Nature* release; the coloured bar is what proofreading added
since. The deep types gained least in absolute terms because they had least
axon in the volume to begin with — an L6CT axon leaves through the white
matter — while L4 and L2/3 gained the most.

::: {.aside-note}
**There are a total 1,834,067 logged manual edit operations** — made
by 152 people. This is an incredible effort that absolutely will not scale to larger connectomes. 
However, all of the edit information is released publicly as training data and informs model advances in autoproofreading.
:::

## 03 — Direct our proofreading

The column effort is winding down; the proofreading effort is not. VORTEX takes requests for which
cells to proofread next, from anyone with a question that needs a cleaner
reconstruction than the automated segmentation gives.

[**Submit a VORTEX request** → www.microns-explorer.org/vortex](https://www.microns-explorer.org/vortex)

## 04 — A cortical column, every neuron proofread

A 100 µm column through all layers of V1, in which **every** neuron has been
proofread rather than a sample. That is what makes the analyses in section 05
possible: a connection that is absent is really absent, not merely untraced.

### Every arbor in the column

![](figures/block04b_hero_row.png){fig-alt="Left: all 1,168 excitatory arbors
drawn in place, coloured by cell type, 1,425 by 570 micrometres. Centre: soma
positions for excitatory and inhibitory cells. Right: 126 astrocytes."}

All 1,168 excitatory neurons with a cell-type label, every dendrite and axon
drawn in place and coloured by `meso_type`. Line width is Strahler order, so
trunks read heavier than tips; dendrite is drawn 1.5× the axon's width at the
same order and the axon is the same hue darkened.

The somata occupy a patch about 130 µm across. The arbors span **1,425 µm**.
That gap is the whole argument for the next two sections: a column is a small
object whose output is not.

The grey behind the soma panels is the untracked population — 132,224 of the
volume's 134,569 cells — binned by soma density.

### Where the cells are, by type

The interactive version of the soma panels above. **Click a type in the legend
to hide it; double-click one to isolate it.** Hover gives the depth and the
nucleus id.

In [3]:
#| echo: false
#| label: fig-somata
#| fig-cap: "Soma positions of every labelled cell in the column, depth against the medial–lateral axis. Excitatory types are circles, inhibitory families diamonds."
#| code-summary: "Code: interactive soma positions"
# Equal unit scale on both axes, so a micrometre across is a micrometre down.
# The column is 133 µm wide and 705 µm deep, so that is a 1:5.3 picture and
# the plot has to be tall — drawn any other shape it says the column is
# wider than it is.
PAD = 12.0                       # µm of air around the data
x0, x1 = somata.um_x.min() - PAD, somata.um_x.max() + PAD
y0, y1 = somata.depth_um.min() - PAD, somata.depth_um.max() + PAD

fig = go.Figure()
for group in GROUPS:
    part = somata[somata.group == group]
    excitatory = group in EXCITATORY
    fig.add_trace(go.Scattergl(
        x=part.um_x, y=part.depth_um,
        mode="markers",
        name=f"{SHOWN.get(group, group)} ({len(part)})",
        legendgroup="excitatory" if excitatory else "inhibitory",
        legendgrouptitle_text=("Excitatory (meso type)" if excitatory
                               else "Inhibitory (family)"),
        marker=dict(size=7 if excitatory else 9,
                    color=COLOUR.get(group, "#777777"),
                    symbol="circle" if excitatory else "diamond",
                    line=dict(width=0.6, color="white")),
        customdata=np.stack([part.meso_type, part.nucleus_id,
                             part.um_z], axis=-1),
        hovertemplate=("<b>%{customdata[0]}</b><br>"
                       "depth %{y:.0f} µm<br>"
                       "medial–lateral %{x:.0f} µm<br>"
                       "anterior–posterior %{customdata[2]:.0f} µm<br>"
                       "nucleus %{customdata[1]}<extra></extra>")))

fig.update_layout(**layout(
    height=900,
    title="Soma positions in the proofread column",
    # `constrain="domain"` is what makes the equal scale narrow the *drawing*
    # rather than widen the *range*: without it Plotly satisfies the
    # constraint by padding the x axis out to several hundred empty
    # micrometres, which looks like a wide column rather than a narrow one.
    xaxis=dict(title="Medial ←→ Lateral (µm)", range=[x0, x1],
               constrain="domain"),
    # Explicit reversed range rather than autorange="reversed": an autoranged
    # axis and a scale constraint disagree about who decides the range.
    yaxis=dict(title="Depth (µm)", range=[y1, y0],
               scaleanchor="x", scaleratio=1),
    legend=dict(groupclick="toggleitem", y=1, x=1.02, xanchor="left"),
))
fig.show(config=CONFIG)



### One cell of each type

An example of each major cell class reconstructed within the column. The neurons all share one scale and alignment, the glia have their own

![](figures/block04f_rogues_gallery.png){fig-alt="A row of individual
reconstructed neurons, one per cell type, each drawn at the same scale."}

To see the morphology in detail, **Click any cell to open it at its native scale**, then use the arrows to step through all of them.

::: {layout-ncol=4}
![L2 IT](figures/block04g_solo_l2it.png){.lightbox group="exemplars" description="Layer 2 intratelencephalic" fig-alt="Layer 2 intratelencephalic, reconstructed"}

![L3 IT](figures/block04g_solo_l3it.png){.lightbox group="exemplars" description="Layer 3 intratelencephalic" fig-alt="Layer 3 intratelencephalic, reconstructed"}

![L4 IT](figures/block04g_solo_l4it.png){.lightbox group="exemplars" description="Layer 4 intratelencephalic" fig-alt="Layer 4 intratelencephalic, reconstructed"}

![L5 IT](figures/block04g_solo_l5it.png){.lightbox group="exemplars" description="Layer 5 intratelencephalic" fig-alt="Layer 5 intratelencephalic, reconstructed"}

![L5 ET](figures/block04g_solo_l5et.png){.lightbox group="exemplars" description="Layer 5 extratelencephalic" fig-alt="Layer 5 extratelencephalic, reconstructed"}

![L5 NP](figures/block04g_solo_l5np.png){.lightbox group="exemplars" description="Layer 5 near-projecting" fig-alt="Layer 5 near-projecting, reconstructed"}

![L6 IT](figures/block04g_solo_l6it.png){.lightbox group="exemplars" description="Layer 6 intratelencephalic" fig-alt="Layer 6 intratelencephalic, reconstructed"}

![L6 CT](figures/block04g_solo_l6ct.png){.lightbox group="exemplars" description="Layer 6 corticothalamic" fig-alt="Layer 6 corticothalamic, reconstructed"}

![Basket](figures/block04g_solo_basket.png){.lightbox group="exemplars" description="Basket cell, putative PV subtype" fig-alt="Basket cell, putative PV subtype, reconstructed"}

![Martinotti](figures/block04g_solo_martinotti.png){.lightbox group="exemplars" description="Martinotti cell, putative SST subtype" fig-alt="Martinotti cell, putative SST subtype, reconstructed"}

![Bipolar](figures/block04g_solo_bipolar.png){.lightbox group="exemplars" description="Bipolar cell, putative VIP subtype" fig-alt="Bipolar cell, putative VIP subtype, reconstructed"}

![Neurogliaform](figures/block04g_solo_ngc.png){.lightbox group="exemplars" description="Neurogliaform cell" fig-alt="Neurogliaform cell, reconstructed"}

![Chandelier](figures/block04g_solo_chandelier.png){.lightbox group="exemplars" description="Chandelier cell" fig-alt="Chandelier cell, reconstructed"}

![Astrocyte](figures/block04g_solo_astrocyte.png){.lightbox group="exemplars" description="Astrocyte" fig-alt="Astrocyte, reconstructed"}

![Microglia](figures/block04g_solo_microglia.png){.lightbox group="exemplars" description="Microglial cell" fig-alt="Microglial cell, reconstructed"}

![Oligodendrocyte](figures/block04g_solo_oligo.png){.lightbox group="exemplars" description="Oligodendrocyte" fig-alt="Oligodendrocyte, reconstructed"}
:::


## 05 — What analysis does a dense column enable?

### Directed connectivity graph

![](figures/block05e_adjacency.png){fig-alt="A 1,329 by 1,329 adjacency matrix,
presynaptic down and postsynaptic across, grouped by cell type with a colour
bar on two margins."}

Every connection between the labelled column cells: presynaptic down,
postsynaptic across, grouped by type and ordered by soma depth within type, so
the diagonal is the column's depth axis. Mark size and ink both carry synapse
count.

The structure you can read off it without counting anything: the inhibitory
rows are dense with outputs and the inhibitory columns are dense with inputs across layers, basket and Martinotti
most of all. There is a conspicuous column at L5ET, representing the dense inputs across layers. 
Excitatory-excitatory connectivity is sparse and most often 1 synapse per connections. 
Recurrent connections dominate in layers 2, 3, and 4. The numbers behind
that texture are in the next figure.

::: {.aside-note}
**This table is complete, not a sample.** The column's internal synapses were
queried with *both* partners restricted to the column set, so a blank cell is
a real absence rather than an untraced one. 82,614 connections, 152,336
synapses, 4.7% of ordered pairs connected. Of the pairs connected at all,
**14.4% are reciprocal — six times what a random graph of this density gives.**
:::

### Type to type, as proportions

The matrix as numbers. A dot's **area** is the proportion of one type's local
synapses that the pair accounts for; the left panel normalises each row (of
everything this type sends locally, where does it go) and the right normalises
each column (of everything this type receives locally, where is it from).

**Click a type in the legend to drop its row or column.** Hover gives the
synapse count and both denominators.

In [4]:
#| echo: false
#| label: fig-dots
#| fig-cap: "Type-to-type connectivity as proportions of local synapses. Top: each row sums to 1 across that type's local output. Bottom: each column sums to 1 across that type's local input. One size scale across both panels."
#| code-summary: "Code: interactive type-to-type dot plots"
present = [g for g in ORDER if g in set(types.pre)]
index = {g: i for i, g in enumerate(present)}
shown = [SHOWN.get(g, g) for g in present]

# One scale across both panels. The largest row-normalised entry is 0.41 and
# the largest column-normalised 0.53; scaling each panel to its own maximum
# would draw those two as the same dot and make the panels incomparable.
vmax = float(max(types.out_share.max(), types.in_share.max()))
SIZE_MAX = 30.0          # marker diameter in px for a dot at `vmax`

# Stacked, not side by side. Thirteen type labels per axis need about 430 px
# to stay apart, and two panels of that will not fit the body measure — which
# is what sent this figure into the page column, and out past the margins.
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.13,
                    subplot_titles=("Local <b>output</b> — where it goes",
                                    "Local <b>input</b> — where it comes from"))

for panel, (column, colour_by) in enumerate(
        [("out_share", "pre"), ("in_share", "post")], start=1):
    for group in present:
        # Colour by the normalised direction, as the static figure does: the
        # output panel comes out in horizontal bands and the input panel in
        # vertical ones, which is the quickest way to see which is which.
        part = types[types[colour_by] == group]
        part = part[part[column] > 0]
        if not len(part):
            continue
        # Area proportional to the share, so diameter goes as its square root.
        diameter = SIZE_MAX * np.sqrt(np.clip(part[column] / vmax, 0, 1))
        fig.add_trace(go.Scatter(
            x=[index[p] for p in part.post],
            y=[index[p] for p in part.pre],
            mode="markers",
            name=SHOWN.get(group, group),
            legendgroup=group,
            showlegend=(panel == 1),
            marker=dict(size=diameter, sizemode="diameter",
                        color=COLOUR.get(group, "#777777"),
                        line=dict(width=1, color="white"), opacity=0.92),
            customdata=np.stack([part.pre, part.post, part.synapses,
                                 part.out_share * 100, part.in_share * 100],
                                axis=-1),
            hovertemplate=("<b>%{customdata[0]} → %{customdata[1]}</b><br>"
                           "%{customdata[2]:,} synapses<br>"
                           "%{customdata[3]:.1f}% of the source's local output"
                           "<br>%{customdata[4]:.1f}% of the target's local "
                           "input<extra></extra>")),
            row=panel, col=1)

ticks = dict(tickmode="array", tickvals=list(range(len(present))),
             ticktext=shown)
for panel in (1, 2):
    # Only the lower panel names the x axis: stacked, the upper one's title
    # sits directly under the lower one's subplot title.
    fig.update_xaxes(title_text="Postsynaptic" if panel == 2 else None,
                     **ticks, tickangle=-45, row=panel, col=1)
    fig.update_yaxes(title_text="Presynaptic", autorange="reversed",
                     **ticks, row=panel, col=1)

# The right margin is what the legend sits in, so it has to be wide
# enough for the longest entry ("Martinotti") plus its swatch.
fig.update_layout(**layout(
    height=1180, margin=dict(l=110, r=165, t=70, b=90),
    legend=dict(title="Type", y=1, x=1.01, xanchor="left")))
fig.show(config=CONFIG)


In [5]:
#| label: dots-readings
#| echo: false
#| code-summary: "Code: the three readings, computed rather than typed"
wide_out = types.pivot(index="pre", columns="post", values="out_share")
wide_in = types.pivot(index="pre", columns="post", values="in_share")
excit = [g for g in EXCITATORY if g in wide_out.index]
inhib = [g for g in wide_out.index if g not in EXCITATORY]

to_inhibitory = wide_out.loc[excit, inhib].sum(axis=1) * 100
from_bc_mc = wide_in.loc[["Basket family", "Martinotti family"],
                         excit].sum() * 100
local_out = (totals.local_out.sum() / totals.all_out.sum())
e_out = totals.loc[excit].local_out.sum() / totals.loc[excit].all_out.sum()
i_out = totals.loc[inhib].local_out.sum() / totals.loc[inhib].all_out.sum()
e_in = totals.loc[excit].local_in.sum() / totals.loc[excit].all_in.sum()
i_in = totals.loc[inhib].local_in.sum() / totals.loc[inhib].all_in.sum()

print(f"excitatory -> interneurons : {to_inhibitory.min():.0f}–"
      f"{to_inhibitory.max():.0f}% of local output")
print(f"basket + Martinotti -> each excitatory type : "
      f"{from_bc_mc.min():.0f}–{from_bc_mc.max():.0f}% of local input")
print(f"local share of output : excitatory {e_out:.1%}, inhibitory {i_out:.1%}")
print(f"local share of input  : excitatory {e_in:.1%}, inhibitory {i_in:.1%}")

excitatory -> interneurons : 29–58% of local output
basket + Martinotti -> each excitatory type : 41–71% of local input
local share of output : excitatory 10.0%, inhibitory 19.0%
local share of input  : excitatory 3.1%, inhibitory 4.2%


Three things the two panels say together:

- **Local excitatory output is mostly inhibitory recruitment.** Every
  excitatory type sends between 29% and 58% of its local output to
  interneurons — L6CT puts 41% into Martinotti cells alone.
- **Each type's heaviest excitatory target sits near its own depth.** L6CT
  sends 29% of its local output to L6CT and 1.0% to L2IT. The diagonal of the
  excitatory block is the column's depth axis.
- **Input is inhibition-dominated everywhere.** Basket plus Martinotti supply
  between 41% and 71% of every excitatory type's local input.

::: {.aside-note}
**"Local" is carrying all three of those.** Every proportion is of the
synapses *inside the column*, and that is a small and uneven slice: excitatory
cells place 10.0% of their output inside the column and receive 3.1% of their
input from it; inhibitory cells 19.0% and 4.2%. So an inhibitory row is a
proportion of twice as large a slice as an excitatory one. The reason is
section 06 — excitatory axons leave, inhibitory axons largely do not.
:::

### Not on the poster: type-to-type as a static figure

![](figures/block05f_dots.png){fig-alt="The two dot-plot panels as rendered for
print, with synapse totals beside each row and above each column."}

The print version of the interactive figure above, with the denominators
written beside each row and above each column. 

### Modest expansion reveals reciprocal motifs

![](figures/block05b_halo_reciprocal.png){fig-alt="Four panels: the inhibitory
halo in section and from above, then basket-to-L5IT reciprocal pairs at v1300
and at v1926."}

The column's cells were proofread first; the expanded ring of interneurons around
them came later. Adding 213 inhibitory cells to the 160 already tracked takes
basket ↔ 5P-IT reciprocal pairs from 52 to 134.

::: {.aside-note}
**That is not proofreading finding more synapses within the core column.** Of the 82 extra pairs,
**76 required a cell that was not in the core set at all** and only **6**
are new links between cells that were already there. The series is a statement
about how much more of the circuit is visible with modest expansion, a nonlinear payoff for effort, not about the circuit itself.
:::

### Functional unit, with coregistration

![](figures/block05c_coreg_added.png){fig-alt="Top: the coregistered field from
above with the worked column cells marked. Bottom: a density of the added
connections, binned at the recipient."}

A coregistered cell is one matched to the two-photon imaging, so a
coreg → coreg connection is the unit a structure–function analysis can use.
The coregistered population does not change between versions — 15,439 cells at both v1300
and v1926 — so nothing here is relabelling, only an increase in proofread connections

336 coregistered column cells gained more than five output synapses between the versions.
Those cells put **28,844 new connections** onto **10,498 recipients**, binned
here at the recipient rather than the source. 336 cells in a 130 µm patch reach most of a
1,160 µm sheet.

### Not on the poster: the column from above

![](figures/block05a_column_topdown.png){fig-alt="The column's arbors seen from
above, lateral against anterior-posterior, with the EM volume outlined."}

The dashed box is the EM volume. The volume is about 1,250 µm
medial-to-lateral and only about **530 µm** anterior-to-posterior, so the
cloud reaching the top and bottom edges is the edge of the tissue, not the
arbors naturally terminating. Whereas many arbors naturally terminate in the lateral dimension, as seen in 06.



## 06 — Interareal connections

### Column axons that reach a higher visual area

![](figures/block06a_reach_fan.png){fig-alt="Column axons crossing from VISp
into VISrl, coloured by cell type, with a density of output synapses along the
medial-lateral axis beneath."}

The volume spans the V1 border, so a column axon that projects to a higher
visual area can be followed into it. Grey is cable that stays in V1; coloured
cable has crossed the boundary at 1,054 µm into VISrl.

Beneath, the same cells' output synapses as a density along the
medial–lateral axis. The distal bump is not a shoulder of the local peak: it
is a patch, 132 µm past the boundary, and 85% of the distal output falls
within 225 µm of it.

::: {.aside-note}
**"Retinotopically matched" is approximate as measured from the in vivo retinotopic sign-mapping.** The
precise position of that patch is drawn by this figure as the natural reading of the axon projections.
:::

### How far the output goes, by type

The density curve above, split by the source cell's type. **Click a type in
the legend to hide it.**

In [6]:
#| echo: false
#| label: fig-reach
#| fig-cap: "Output synapses per micrometre along the medial–lateral axis, by presynaptic type. The dashed line is the V1/VISrl boundary; everything to its right is interareal."
#| code-summary: "Code: interactive reach density"
boundary = reach_meta["boundary_um"]

fig = go.Figure()
for group in [g for g in ORDER if g in set(density.group)]:
    part = density[density.group == group].sort_values("um_x")
    fig.add_trace(go.Scatter(
        x=part.um_x, y=part.per_um,
        mode="lines", name=f"{SHOWN.get(group, group)}",
        line=dict(color=COLOUR.get(group, "#777777"), width=2.2),
        customdata=part.synapses,
        hovertemplate=("<b>%{fullData.name}</b><br>"
                       "%{x:.0f} µm<br>%{y:.1f} synapses/µm<br>"
                       "%{customdata:,} in this 25 µm bin<extra></extra>")))

fig.add_vline(x=boundary, line=dict(color="#5B6368", width=1.6, dash="dash"))
fig.add_annotation(x=boundary, y=1.0, yref="paper", yanchor="bottom",
                   text="V1 │ VISrl", showarrow=False,
                   font=dict(size=13, color="#5B6368"))

fig.update_layout(**layout(
    height=560,
    title="Where the column's output synapses land",
    xaxis_title="Medial ←→ Lateral (µm)",
    yaxis_title="Output synapses per µm",
    hovermode="x unified",
    legend=dict(title="Presynaptic type", y=1, x=1.02),
))
fig.show(config=CONFIG)


### Layer 5 on its own

![](figures/block06b_layer5_fan.png){fig-alt="The same reach figure restricted
to layer 5 cells: ET, IT and NP combined."}

Layer 5 — ET, IT and NP together — crosses the boundary more often than the
column as a whole: **91 of 135 cells (67%)** against 560 of 1,168 (48%). But
it places a *smaller* share of its synapses past the line, 5.6% against 6.6%.
More L5 cells reach the higher area; each one puts less there.

## 07 — Connectome in context

![](figures/panel07_glia.png){fig-alt="Left: an oligodendrocyte with an inset
showing myelin around an axon in the raw EM. Right: a microglial cell with the
synapses it contacts marked."}

The same volume holds the non-neuronal cells, densely segmented along with the neurons. 
A set of astrocytes, oligodendrocytes, and microglia within the column have undergone 
proofreading to clean and complete their processes. On the left an oligodendrocyte, 
with an inset from the EM imagery showing myelin
wrapped around the axon it ensheathes. On the right a microglial cell, with
every synapse within 500 nm of its processes marked — 4,648 of them.

Neither is part of the connectome as usually defined, but here the non-neuronal cells
are in the same reconstruction with the neurons they support and synapses they modify.

### Also in the column: 126 astrocytes

On the printed board this render sits in block 04, beside the neurons it was
reconstructed alongside. It belongs here, with the rest of the glia.

![](figures/block04h_astrocytes.webp){width=330 fig-alt="126 astrocytes from
the column, each a different colour, stacked into a tall column that follows
the same rightward kink as the neuron somata." .lightbox
description="The column's 126 proofread astrocytes, one colour each."}

Every astrocyte whose soma falls in the column, 126 of them, each a separate
colour. **Click to open it larger** — even then the cells read as territories
rather than as individual arbors; the single astrocyte in block 04's
gallery is the one to look at for morphology.

Two things come out of seeing the population rather than one cell. The
territories read as discrete — each astrocyte a patch of colour meeting its
neighbours rather than mixing through them — which is what astrocyte tiling
looks like, though nothing here measures the overlap. And the stack carries
the same rightward kink as the neuronal column in block 04, because it is the
same piece of tissue: the column is defined on the neurons, and the
astrocytes inside it inherit its shape.


## 08 — Organelle segmentation

![](figures/panel08_organelle.png){fig-alt="A neuron with its internal
mitochondria segmented, each a different colour."}

The resolution required for segmenting synapses also yields segmentable organelles, such as mitochondria.
Here, mitochondria inside one neuron, each as a separate object.

![](figures/panel08_mito.png){fig-alt="Four bar charts comparing mitochondrial
content between excitatory and inhibitory cells, in axon and dendrite."}

Mitochondrial content differs between excitatory and inhibitory cells and
between compartments, in every measure: mean mitochondrion size, the fraction
of compartment volume they occupy, their number per micrometre of axon, and
their number per output synapse. Inhibitory axons carry more and larger
mitochondria than excitatory axons. However excitatory dendrites show larger and more numerous mitochondria. 
This is especially apparent in large apical dendrites.

*These results are from a 40 x 40 x 80 um test volume provided by Constantin Pape et al. Full-volume segmentation is in
progress.*

## Where these numbers come from

Every figure on this page is either a panel from the printed poster or an
interactive version of one, built from the same exported numbers.

| | |
|---|---|
| Dataset | MICrONS `minnie65_public` |
| Final materialization | **v1926** |
| Baseline for "since publication" | **v1300**, the *Nature* 2025 release |
| Column definition | `in_v1_column`, 1,355 tracked neurons |
| Labelled subset used by every by-type figure | 1,329 cells with a `meso_type` |
| Cell-type vocabulary | somatodendritic `multifeature` classes and families |
| Coordinates | layer-aligned micrometres via `standard_transform`, pia at depth 0 |


### Acknowledgements

Funded by NIH U24NS120053. Data access and tutorials at
[tutorial.microns-explorer.org](https://tutorial.microns-explorer.org).